# 全國教師在職進修資訊網 - 線上課程蒐集 (Google Colab 版)
本腳本將本地端的所有爬蟲、過濾、與 LLM 驗證程式碼全部遷移至 Colab 執行。
請依序點擊每一個儲存格左側的「播放(Play)」按鈕即可。

### 步驟 1: 環境安裝 (約需 1 分鐘)

In [ ]:
!apt-get update -y
!apt-get install -y xvfb
!pip install -q playwright nest-asyncio
!playwright install chromium
!playwright install-deps

print("環境安裝完成！")


### 步驟 2: 啟動虛擬螢幕模組與設定參數
這裡可以設定您想要爬取的參數 (天數、API Key 等)。

In [ ]:
import os
import datetime

# 啟動 Xvfb 並設定虛擬螢幕變數
os.system('Xvfb :99 -screen 0 1024x768x24 &')
os.environ['DISPLAY'] = ':99'

# @markdown ### 📅 搜尋範疇與關鍵字設定
START_DATE = "" # @param {type:"date"}
# 若未指定日期，則預設為執行的當天 (today)
if not START_DATE:
    START_DATE = datetime.date.today().strftime('%Y-%m-%d')

SEARCH_DAYS = 3 # @param {type:"slider", min:1, max:30, step:1}
IT_KEYWORDS = ['資訊科技', 'AI', '人工智慧', 'Canva', '程式設計', '數位', '資安', '軟體', '電腦', '網路', '機器人']

print(f"📌 參數設定完成：預計從 {START_DATE} 開始抓取未來 {SEARCH_DAYS} 天內的課程。")

虛擬螢幕已啟動，參數設定完畢。


### 步驟 3: 執行 Playwright 爬蟲 (抓取所有研習課程)

In [ ]:
import asyncio
import nest_asyncio
import time
import csv
import re
from datetime import datetime, timedelta
from playwright.async_api import async_playwright

nest_asyncio.apply() # 確保套件適用於 Jupyter 的 asyncio

def parse_taiwan_date(date_str):
    match_cn = re.search(r'(\d+)年(\d+)月(\d+)日', date_str)
    if match_cn:
        y, m, d = match_cn.groups()
        return datetime(int(y), int(m), int(d))
    match_sl = re.search(r'(\d+)/(\d+)/(\d+)', date_str)
    if match_sl:
        y, m, d = match_sl.groups()
        return datetime(int(y), int(m), int(d))
    return None

all_data = []
filtered_final = []
start_date = datetime.strptime(START_DATE, '%Y-%m-%d')
target_date = start_date + timedelta(days=SEARCH_DAYS)
target_date = target_date.replace(hour=23, minute=59, second=59)

async def scrape_courses():
    global all_data, filtered_final
    max_pages = 200 # 提高爬取頁數上限，避免因為目標日期較遠而在中途提早停止

    async with async_playwright() as p:
        # 在 Colab 中，透過 Xvfb 加持，這裡可以開 headless=False 而不會報錯！
        browser = await p.chromium.launch(headless=False, args=['--disable-blink-features=AutomationControlled', '--no-sandbox', '--disable-setuid-sandbox', '--disable-dev-shm-usage'])
        context = await browser.new_context(user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36')
        page = await context.new_page()
        print(f"正在網頁中搜尋 {start_date.strftime('%Y/%m/%d')} 至 {target_date.strftime('%Y/%m/%d')} 的所有課程...")

        # Add retry logic for page.goto()
        max_retries = 3
        for i in range(max_retries):
            try:
                print(f"嘗試導航至網頁 (第 {i+1} 次)...")
                await page.goto("https://www2.inservice.edu.tw/main2-3.aspx", timeout=60000) # Increased timeout to 60 seconds
                break # If successful, break the loop
            except Exception as e:
                print(f"導航失敗: {e}")
                if i < max_retries - 1:
                    print("等待 5 秒後重試...")
                    await asyncio.sleep(5)
                else:
                    raise # Re-raise the exception if all retries fail

        try:
            await page.wait_for_selector("select", timeout=15000)
            select_box = page.locator("select").filter(has_text="50").first
            await select_box.select_option("100")
            await page.wait_for_load_state("networkidle")
            await asyncio.sleep(3)
        except Exception as e:
            pass # Ignored timeout

        page_num = 1
        while page_num <= max_pages:
            await page.wait_for_selector("#MasterGrid", timeout=20000)
            rows = await page.eval_on_selector_all("#MasterGrid tr", """
                (trs) => trs.map(tr => {
                    return Array.from(tr.querySelectorAll('th, td')).map(td => td.innerText.trim().replace(/[\\n,]/g, ' '));
                })
            """)

            if page_num == 1:
                all_data.append(rows[0])

            data_rows = rows[1:-1]
            if not data_rows: break

            all_data.extend(data_rows)

            last_date_str = "Unknown"
            current_date = None
            # 從最後面倒回去找，避開含有隱藏資訊的備註列(通常沒有完整的 5 個欄位)
            for row in reversed(data_rows):
                if len(row) > 4:
                    parsed = parse_taiwan_date(row[4])
                    if parsed:
                        last_date_str = row[4]
                        current_date = parsed
                        break
            print(f"...已爬取第 {page_num} 頁 ({len(data_rows)}筆)，目前最後日期: {last_date_str}")

            if current_date and current_date > target_date:
                break

            next_btn = page.get_by_role("button", name="下一頁")
            if await next_btn.is_visible() and await next_btn.is_enabled():
                first_code = data_rows[0][1] if len(data_rows[0]) > 1 else ""
                await next_btn.click()
                try:
                    await page.wait_for_function(f"document.querySelector('#MasterGrid tr:nth-child(2) td:nth-child(2)')?.innerText.trim() !== '{first_code}'", timeout=15000)
                except:
                    await asyncio.sleep(3)
                page_num += 1
            else:
                break

        await browser.close()

    filtered_final.append(all_data[0])
    for r in all_data[1:]:
        if len(r) > 4:
            d = parse_taiwan_date(r[4])
            if d and start_date <= d <= target_date:
                filtered_final.append(r)

    print(f"\n爬蟲完畢！在此區間內共找到 {len(filtered_final)-1} 筆課程。")

await scrape_courses()


正在網頁中搜尋 2026/04/19 至 2026/04/22 的所有課程...
嘗試導航至網頁 (第 1 次)...
...已爬取第 1 頁 (200筆)，目前最後日期: 2026年4月14日
...已爬取第 2 頁 (200筆)，目前最後日期: 2026年4月15日
...已爬取第 3 頁 (200筆)，目前最後日期: 2026年4月15日
...已爬取第 4 頁 (200筆)，目前最後日期: 2026年4月15日
...已爬取第 5 頁 (200筆)，目前最後日期: 2026年4月15日
...已爬取第 6 頁 (200筆)，目前最後日期: 2026年4月15日
...已爬取第 7 頁 (200筆)，目前最後日期: 2026年4月15日
...已爬取第 8 頁 (200筆)，目前最後日期: 2026年4月15日
...已爬取第 9 頁 (41筆)，目前最後日期: 2026年4月15日
...已爬取第 10 頁 (200筆)，目前最後日期: 2026年4月15日
...已爬取第 11 頁 (200筆)，目前最後日期: 2026年4月16日
...已爬取第 12 頁 (200筆)，目前最後日期: 2026年4月16日
...已爬取第 13 頁 (200筆)，目前最後日期: 2026年4月17日
...已爬取第 14 頁 (200筆)，目前最後日期: 2026年4月17日
...已爬取第 15 頁 (200筆)，目前最後日期: 2026年4月18日
...已爬取第 16 頁 (200筆)，目前最後日期: 2026年4月18日
...已爬取第 17 頁 (200筆)，目前最後日期: 2026年4月18日
...已爬取第 18 頁 (200筆)，目前最後日期: 2026年4月19日
...已爬取第 19 頁 (200筆)，目前最後日期: 2026年4月21日
...已爬取第 20 頁 (200筆)，目前最後日期: 2026年4月21日
...已爬取第 21 頁 (200筆)，目前最後日期: 2026年4月22日
...已爬取第 22 頁 (200筆)，目前最後日期: 2026年4月22日
...已爬取第 23 頁 (200筆)，目前最後日期: 2026年4月22日
...已爬取第 24 頁 (200筆)，目前最後日期: 2026年4月22日
...已爬取第 25 頁 (

### 步驟 4: 主題過濾 (挑出 IT/AI 相關課程)

In [ ]:
try:
    from google.colab import ai
except ImportError:
    print("無法匯入 google.colab.ai。提醒：這只能在 Google Colab 環境下執行。")

import json
import re

it_courses = [filtered_final[0]] # header
courses_to_check = [row for row in filtered_final[1:] if len(row) > 2]
print(f"準備對 {len(courses_to_check)} 筆課程進行 LLM 主題過濾...")

# 每次處理 50 筆，避免 LLM 偷懶或超過字數限制
chunk_size = 50
for i in range(0, len(courses_to_check), chunk_size):
    chunk = courses_to_check[i:i+chunk_size]

    course_list_text = ""
    for j, row in enumerate(chunk):
        course_list_text += f"{j}. {row[2]}\n"

    prompt = f"""在列表中逐一掃描研習名稱，找出與以下主題相關的課程：
- 資訊科技、程式設計
- AI 相關 (例如：Gemini, NotebookLM 等，但不限於列出的這些)
- Canva 設計
- 機電整合、生活科技
- STEAM

請嚴格以純 JSON 格式回傳符合條件的「課程編號」陣列，例如：[0, 2, 4]。不要加上 ```json 標籤或是任何其他解釋！
如果沒有符合的課程，請回傳 []。

課程列表如下：
{course_list_text}
"""
    try:
        print(f"傳送第 {i+1} ~ {min(i+chunk_size, len(courses_to_check))} 筆至 LLM 分析...")
        response = ai.generate_text(prompt)
        response_text = str(response).replace('`json', '').replace('`', '').strip()

        # 嘗試用 Regex 抓取陣列結構
        json_match = re.search(r'\[.*\]', response_text, re.DOTALL)
        if json_match:
            indices = json.loads(json_match.group(0))
        else:
            indices = json.loads(response_text)

        if not isinstance(indices, list):
            raise ValueError("LLM 沒有回傳有效的陣列格式。")

        for idx in indices:
            if isinstance(idx, int) and 0 <= idx < len(chunk):
                it_courses.append(chunk[idx])
                print(f"  ✅ 保留 [{i+idx}]: {chunk[idx][2]}")

    except Exception as e:
        print(f"  ! LLM 解析失敗 (該批次全部預設保留): {e}")
        try:
             print(f"  > LLM 回應結果：{response_text[:100]}...")
        except: pass
        it_courses.extend(chunk)

print(f"\n經過 LLM 過濾，符合上述主題的課程有 {len(it_courses)-1} 筆。")


準備對 682 筆課程進行 LLM 主題過濾...
傳送第 1 ~ 50 筆至 LLM 分析...
  ✅ 保留 [0]: 紅葉國小－數位學習精進方案－全縣數位學習公開課
  ✅ 保留 [21]: [精進數位]「115年推動中小學數位學習精進方案」重點學校公開觀課
  ✅ 保留 [23]: [精進數位]B5-1生成式AI與教育應用工作坊
  ✅ 保留 [24]: [精進數位]B5-1生成式AI與教育應用工作坊
  ✅ 保留 [29]: 「Canva AI行政效率大提升」增能研習
  ✅ 保留 [30]: 「Canva融入跨領域課程設計」
  ✅ 保留 [39]: 115.04.20中央輔導團科技領域分團【數位點亮中小科技新課堂-金門縣科技領域分團點燈】(線上)
  ✅ 保留 [40]: 115.04.20科技領域分團-分團工作會議暨課程分享增能工作坊
  ✅ 保留 [44]: 115學年度宜蘭區實用技能學程分發說明會及系統操作研習
  ✅ 保留 [45]: A.I.國寫一條龍
  ✅ 保留 [46]: AI與SEL融入國語文教學&學習歷程工作坊
  ✅ 保留 [47]: Test My English 英檢系統高中職線上研習_系統功能介紹與開口表達練習全指南_場次一
傳送第 51 ~ 100 筆至 LLM 分析...
  ✅ 保留 [61]: 基隆市百福科技中心辦理「Arduino 初階」線上教師研習 (1)
  ✅ 保留 [63]: 教育部115年沉浸科技導入素養導向教學實施計畫-EWova教育元宇宙平臺教師增能工作坊
  ✅ 保留 [71]: 嘉義市114學年度精進課程計畫--國中國文領域「數位及AI工具輔助讀寫教學」
  ✅ 保留 [79]: [防災教育]南投縣115年防災教育輔導團增能研習實施計畫 —運用無人機科技於校園災害防救
  ✅ 保留 [83]: [素養導向]【114精進計畫-科技領域輔導團】教學實踐輔導(4)
  ✅ 保留 [91]: [素養導向]AI 畫筆下的數學魔術物語
  ✅ 保留 [92]: [素養導向]一鍵升級教學力：Google AI全攻略（備課×出題×解題）
傳送第 101 ~ 150 筆至 LLM 分析...
  ✅ 保留 [110]: [領綱素養]AI科技在自然教學的應用
  ✅ 保留 [112]: [領綱素養]通霄科技中心【性別

### 步驟 5: 爬蟲二次驗證 (進入課程頁面尋找線上視訊連結)

In [ ]:
import json

valid_results = []

async def verify_courses():
    global valid_results
    if len(it_courses) > 1:
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=False, args=['--disable-blink-features=AutomationControlled', '--no-sandbox', '--disable-setuid-sandbox', '--disable-dev-shm-usage']) # 依然使用無頭防護
            context = await browser.new_context(user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36')
            page = await context.new_page()

            for i, row in enumerate(it_courses[1:]):
                if len(row) < 3: continue
                cid = row[1]
                name = row[2]

                print(f"正在驗證 [{i+1}/{len(it_courses)-1}] {name}...")
                url = f"https://www2.inservice.edu.tw/NAPP/CourseView.aspx?cid={cid}"

                try:
                    await page.goto(url, timeout=30000)
                    await asyncio.sleep(2)

                    page_text = await page.inner_text("body")
                    content = await page.content()

                    is_online = any(term.lower() in page_text.lower() for term in ['meet.google.com', 'teams.microsoft.com', 'zoom.us', 'webex', '數位遠距教學', '線上研習', '視訊連結', 'meet'])

                    has_maps_link = 'google.com/maps' in content or 'maps.google.com' in content
                    onsite_location = False
                    location_text = ""
                    try:
                        if "開課地點：" in page_text:
                            match = re.search(r'開課地點：(.*)', page_text)
                            if match:
                                location_text = match.group(1).strip()
                    except: pass

                    if location_text and any(p in location_text for p in ['路', '號', '里', '區', '電腦教室', '室', '校']):
                        if "線上" not in location_text and "數位" not in location_text:
                            onsite_location = True

                    is_onsite = any(term in page_text for term in ['實體教室', '現場參加', '僅限校內教師參加', '未開放線上報名', '現場報名', '報到地點', '本校教室'])
                    if has_maps_link: is_onsite = True
                    if is_online: is_onsite = False # Meeting link overrides

                    if is_online and not is_onsite:
                        valid_results.append({
                            "cid": cid,
                            "name": name,
                            "time": row[4] if len(row) > 4 else "Unknown",
                            "raw_text": page_text[:2000]
                        })
                        print("  -> 驗證通過：是純線上課程！")
                    else:
                        print("  -> 過濾：實體課程或非線上課程。")

                except Exception as e:
                     print(f"  -> Error checking {cid}: {e}")

            await browser.close()

    print(f"\n第一階段驗證完畢，共有 {len(valid_results)} 堂純線上課程準備交給 Gemini 分析。")

await verify_courses()



正在驗證 [1/135] 紅葉國小－數位學習精進方案－全縣數位學習公開課...
  -> 過濾：實體課程或非線上課程。
正在驗證 [2/135] [精進數位]「115年推動中小學數位學習精進方案」重點學校公開觀課...
  -> 過濾：實體課程或非線上課程。
正在驗證 [3/135] [精進數位]B5-1生成式AI與教育應用工作坊...
  -> 過濾：實體課程或非線上課程。
正在驗證 [4/135] [精進數位]B5-1生成式AI與教育應用工作坊...
  -> 過濾：實體課程或非線上課程。
正在驗證 [5/135] 「Canva AI行政效率大提升」增能研習...
  -> 過濾：實體課程或非線上課程。
正在驗證 [6/135] 「Canva融入跨領域課程設計」...
  -> 過濾：實體課程或非線上課程。
正在驗證 [7/135] 115.04.20中央輔導團科技領域分團【數位點亮中小科技新課堂-金門縣科技領域分團點燈】(線上)...
  -> 驗證通過：是純線上課程！
正在驗證 [8/135] 115.04.20科技領域分團-分團工作會議暨課程分享增能工作坊...
  -> 過濾：實體課程或非線上課程。
正在驗證 [9/135] 115學年度宜蘭區實用技能學程分發說明會及系統操作研習...
  -> 過濾：實體課程或非線上課程。
正在驗證 [10/135] A.I.國寫一條龍...
  -> 過濾：實體課程或非線上課程。
正在驗證 [11/135] AI與SEL融入國語文教學&學習歷程工作坊...
  -> 過濾：實體課程或非線上課程。
正在驗證 [12/135] Test My English 英檢系統高中職線上研習_系統功能介紹與開口表達練習全指南_場次一...
  -> 驗證通過：是純線上課程！
正在驗證 [13/135] 基隆市百福科技中心辦理「Arduino 初階」線上教師研習 (1)...
  -> 驗證通過：是純線上課程！
正在驗證 [14/135] 教育部115年沉浸科技導入素養導向教學實施計畫-EWova教育元宇宙平臺教師增能工作坊...
  -> 過濾：實體課程或非線上課程。
正在驗證 [15/135] 嘉義市114學年度精進課程計畫--國中國文領域「數位及AI工具輔助讀寫教學」...
  -> 過濾：實體課程或非線上課程。
正在驗證 [

### 步驟 6: 呼叫 Gemini AI 精準擷取時間/主講人，生成 Markdown 報告

In [ ]:
# 取代原本需 API key 的寫法，改由 Colab 內建的 ai 模組直接呼叫
try:
    from google.colab import ai
except ImportError:
    print("無法匯入 google.colab.ai。提醒：這只能在 Google Colab 環境下執行。")

import json
from datetime import datetime

target_date_stamp = target_date.strftime('%Y%m%d')
output_file = f"courselist_{target_date_stamp}_final_report.md"

if len(valid_results) == 0:
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(f"# 全國教師在職進修資訊網 - 線上課程清單 ({datetime.now().strftime('%Y/%m/%d')})\n\n")
        f.write("本次範圍內無符合條件之課程。\n")
    print("無符合的課程，已生成空白報表。")
else:
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(f"# 全國教師在職進修資訊網 - 線上課程清單 ({datetime.now().strftime('%Y/%m/%d')})\n\n")
        f.write("> [!NOTE]\n")
        f.write("> **AI 語意分析版本**：本報告由 Colab 內建 google.colab.ai (無須 API key) 擷取資料。\n\n")
        f.write("| 課程代碼 | 課程名稱 | 研習時間 (起訖) | Google Meet 網址 / 線上連結 | 主講人 |\n")
        f.write("| :--- | :--- | :--- | :--- | :--- |\n")

        for i, c in enumerate(valid_results):
            print(f"請 Gemini 分析 [{i+1}/{len(valid_results)}] {c['name']} ...")

            prompt = f"""請作為課程擷取員，從以下文字萃取資訊並返回 JSON 物件。
【提取規則】：
1. time: 完整上課具體起訖時段 (如 2026/04/07(二) 13:20~16:30)，民國年請轉西元年。
2. speaker: 主講人或講師姓名 (拔除身分標籤)，無則填未提供。
3. url: Meet或視訊網址，無則填請見內文。
請嚴格使用不帶任何格式或markdown語法的純 JSON 回傳，格式: {{\"time\": \"...\", \"speaker\": \"...\", \"url\": \"...\"}}

課程原始文字：
{c['raw_text']}
"""
            try:
                # The correct syntax provided by user!
                response = ai.generate_text(prompt)
                response_text = str(response)

                # 清理與擷取 JSON
                response_text = response_text.replace('`json', '').replace('`', '').strip()
                import re
                json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
                if json_match:
                    data = json.loads(json_match.group(0))
                else:
                    data = json.loads(response_text)

                time_str = data.get("time", c.get('time', '未提供'))
                speaker_str = data.get("speaker", "未提供")
                url_str = data.get("url", "請見內文")
            except Exception as e:
                print(f"  ! 擷取失敗: {e}")
                time_str = c.get('time', '未提供')
                speaker_str = "備註：自動擷取失敗"
                url_str = "請見內文"

            official_url = f"https://www2.inservice.edu.tw/NAPP/CourseView.aspx?cid={c['cid']}"
            f.write(f"| [{c['cid']}]({official_url}) | {c['name']} | **{time_str}** | {url_str} | {speaker_str} |\n")

    print(f"\n✅ 報表生成完畢：{output_file}")


請 Gemini 分析 [1/13] 115.04.20中央輔導團科技領域分團【數位點亮中小科技新課堂-金門縣科技領域分團點燈】(線上) ...
請 Gemini 分析 [2/13] Test My English 英檢系統高中職線上研習_系統功能介紹與開口表達練習全指南_場次一 ...
請 Gemini 分析 [3/13] 基隆市百福科技中心辦理「Arduino 初階」線上教師研習 (1) ...
請 Gemini 分析 [4/13] [素養導向]一鍵升級教學力：Google AI全攻略（備課×出題×解題） ...
請 Gemini 分析 [5/13] [雙語教育]中外師與ELTA英語教學增能研習場次六：Integrating Technology: EFL and CLIL ...
請 Gemini 分析 [6/13] 【Cool English】[進階班] 當 AI 成為助教：運用 Cool English 打造高效英語課堂-普技高場次4 ...
請 Gemini 分析 [7/13] 【精進數位】新竹縣政府教育局高中課程素養中心「Gemini & NotebookLM–為學習而打造的Google AI」英文科研習 ...
請 Gemini 分析 [8/13] AI 教學自動化實戰：打造您的專屬助理與高效工作流－114學年度新興科技融入教學創意設計教師專業社群－教育部國教署高中課程美術學科中心 ...
請 Gemini 分析 [9/13] AI人工智慧輔助IA智能增強實驗篇_課程實驗與自然探究Part1 ...
請 Gemini 分析 [10/13] [精進數位]教師數位增能工作坊_用加分吧 x 學習吧進行SEL課程 ...
請 Gemini 分析 [11/13] 「數位閱讀資源」教師實務場 ...
請 Gemini 分析 [12/13] 115年度第1梯次一般教師資訊應用研習-第4場次-【微軟Copilot-AI不只是聊天：Copilot讓你工作效率翻倍！】(線上直播) ...
請 Gemini 分析 [13/13] 玩轉數位課堂-用 Wayground 打造沉浸式趣味教室 ...

✅ 報表生成完畢：courselist_20260414_final_report.md


### 步驟 7: 更新資料回 GitHub
這一步驟可以自動將產出的報表推送到你的 GitHub 專案中。

In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

# 1. 設定 Git 使用者資訊
!git config --global user.email "jefffang.edu@gmail.com"
!git config --global user.name "JeffCodingMentor"

# 2. Clone 專案 (建議使用 GitHub Personal Access Token)
# 格式為 https://<TOKEN>@github.com/JeffCodingMentor/inspage.git
!git clone https://{token}@github.com/JeffCodingMentor/inspage.git

# 3. 將產生的新 md 檔移動/複製到 inspage/data 目錄下
# !cp courselist_*.md inspage/data/

# 4. 提交並推送到 GitHub
%cd inspage
!git add data/*.md
!git commit -m "Auto-update course data from Colab"
!git push origin main